In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import lightgbm as lgb

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

## 14. Error Analysis

Deep dive into model misclassifications (false positives and false negatives) to identify weaknesses.

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)

exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### 14.1 Load Predictions

In [ ]:
# Train LightGBM model
dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
model = lgb.train(
    {"objective": "binary", "metric": "auc", "num_leaves": 31, "learning_rate": 0.1, "verbose": -1, "random_state": 42},
    dtrain, num_boost_round=100, valid_sets=[dval], callbacks=[lgb.early_stopping(15, verbose=False)]
)
preds = model.predict(X_va, num_iteration=model.best_iteration)
pred_class = (preds >= 0.5).astype(int)

### 14.2 High-Confidence Error Profiling

In [ ]:
analysis_df = X_va.copy()
analysis_df["true_label"] = y_va
analysis_df["pred_prob"] = preds
analysis_df["pred_label"] = pred_class

# Filter out correct predictions
false_positives = analysis_df[(analysis_df["true_label"] == 0) & (analysis_df["pred_label"] == 1)]
false_negatives = analysis_df[(analysis_df["true_label"] == 1) & (analysis_df["pred_label"] == 0)]

print(f"Total Validation Samples: {len(X_va):,}")
print(f"False Positives (Normal flagged as Fraud): {len(false_positives):,}")
print(f"False Negatives (Fraud missed by Model):  {len(false_negatives):,}")

### 14.3 Error analysis by Transaction Amount

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(analysis_df[analysis_df["true_label"] == 0]["TransactionAmt"], label="True Legitimate", log_scale=True, ax=ax)
sns.kdeplot(false_positives["TransactionAmt"], label="False Positives", log_scale=True, ax=ax)
sns.kdeplot(false_negatives["TransactionAmt"], label="False Negatives", log_scale=True, ax=ax)
ax.set_title("Transaction Amount Distribution across Error Groups")
ax.set_xlabel("Transaction Amount (USD, log scale)")
ax.legend()
plt.show()

### 14.4 Most Confident False Positives

In [ ]:
print("Top 10 Most Confident False Positives (Model predicted high risk but transaction was legitimate):")
print(false_positives.sort_values("pred_prob", ascending=False)[["TransactionAmt", "card1", "pred_prob"]].head(10))

### 14.5 Most Confident False Negatives

In [ ]:
print("Top 10 Most Confident False Negatives (Model predicted low risk but transaction was fraudulent):")
print(false_negatives.sort_values("pred_prob", ascending=True)[["TransactionAmt", "card1", "pred_prob"]].head(10))